# BTC Sell-off (Oct 29-30, 2026): Financial News Sentiment Analysis
TextBlob polarity per source, emoji-face charts and a PDF report.

**How to use:** run the cells top to bottom (Runtime > Run all). The PDF downloads at the end.

In [ ]:
# Step 1: install libraries and TextBlob language data (run once)
!pip install -q textblob reportlab
import nltk
for pkg in ["punkt", "punkt_tab", "brown", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng", "wordnet", "conll2000"]:
    nltk.download(pkg, quiet=True)
print("Setup complete")


In [ ]:
# Step 2: imports and settings
import io, os
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Arc, Circle
import pandas as pd
from textblob import TextBlob

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import cm
from reportlab.platypus import (Image, PageBreak, Paragraph, SimpleDocTemplate,
                                Spacer, Table, TableStyle)

OUT_DIR = "/content"          # Colab working folder


### Step 3: source summaries and labels

In [ ]:
POS_T, NEG_T = 0.05, -0.05          # polarity thresholds

# ---------------------------------------------------------------------------
# 1) Source key points (condensed, own wording)
# ---------------------------------------------------------------------------
SOURCES = [
    ("Rio Times",
     "Bitcoin slid toward 77,000 dollars after hackers stole about 6.3 billion dollars "
     "from Binance, the largest theft in crypto history. The price broke below a range "
     "that had held for days and weekly losses neared five percent. Withdrawals were "
     "paused on some networks, although billions in stolen assets were frozen. A "
     "hawkish Federal Reserve hold added to the pressure."),
    ("CNBC",
     "Bitcoin fell about four percent as the record Binance hack rattled digital asset "
     "markets. Ether and the BNB token dropped as well. Binance said a compromised "
     "engineer laptop may have enabled unauthorized withdrawals, froze roughly three "
     "billion dollars, and promised to cover all customer losses."),
    ("24/7 Wall St",
     "Bitcoin dropped to a five-week low as forced liquidations wiped out about 1.3 "
     "billion dollars of leveraged long positions. The selloff began before the hack "
     "news, which suggests leverage was already unwinding. The rebound looks weak and "
     "driven by exhausted sellers rather than new buyers."),
    ("CryptoTicker (Oct 8)",
     "Fed minutes pointed to another interest rate hike by year end and the dream of "
     "rate cuts faded. Spot Bitcoin ETFs recorded net outflows and Bitcoin failed "
     "repeatedly at the top of its range, creating a fragile and bearish backdrop."),
    ("CoinStats",
     "A hawkish Fed hold and a risk-off move in tech stocks pushed Bitcoin below 80,000 "
     "dollars. Early reports of a large wallet drain at Binance were unconfirmed and "
     "estimates varied widely. Sentiment is neutral with a cautious bearish tilt, and "
     "Bitcoin remains far below its all-time high."),
    ("Bitcoin Foundation",
     "Bitcoin briefly slipped under 77,000 dollars while 1.49 billion dollars in "
     "positions were liquidated. Binance investigated suspicious outflows and loss "
     "estimates kept shifting. It is unclear whether the exchange trouble truly caused "
     "the drop because the timing does not settle the question."),
    ("Forbes",
     "Bitcoin hit a six-week low near 76,000 dollars as the Binance hack accelerated "
     "losses. Analysts said the decline was already underway and described it as a "
     "leverage flush rather than an exodus, since open interest fell only slightly."),
    ("TradingKey",
     "Restrictive macro conditions and high Treasury yields weakened institutional risk "
     "appetite. Large options settlements added selling pressure and spot demand stayed "
     "subdued. ETF outflows signaled that institutions are de-risking rather than "
     "buying the dip."),
    ("Investing News Network",
     "Bitcoin touched a six-week low as markets weakened after reports of the Binance "
     "security breach. Ether and BNB fell harder than Bitcoin. Binance vowed to cover "
     "the losses while investigators pointed toward a North Korean attacker."),
]

EMOJI = {"Positive": "🙂", "Neutral": "😐", "Negative": "🙁"}
COLOR = {"Positive": "#2e9e5b", "Neutral": "#8a8f98", "Negative": "#d64545"}


def label_for(p):
    return "Positive" if p > POS_T else "Negative" if p < NEG_T else "Neutral"


### Step 4: TextBlob sentiment + emoji-face charts

In [ ]:
def analyse():
    rows = []
    for name, text in SOURCES:
        s = TextBlob(text).sentiment
        rows.append({"source": name, "polarity": round(s.polarity, 3),
                     "subjectivity": round(s.subjectivity, 3),
                     "label": label_for(s.polarity), "text": text})
    return rows


def face_image(kind):
    fig, ax = plt.subplots(figsize=(1, 1), dpi=120)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    face = {"Positive": "#ffd43b", "Neutral": "#ffd43b", "Negative": "#ffb347"}[kind]
    ax.add_patch(Circle((0, 0), 1, fc=face, ec="#b8860b", lw=2))
    ax.add_patch(Circle((-0.35, 0.3), 0.12, fc="#333"))
    ax.add_patch(Circle((0.35, 0.3), 0.12, fc="#333"))
    if kind == "Positive":
        ax.add_patch(Arc((0, -0.05), 1.0, 0.8, theta1=200, theta2=340, lw=3, color="#333"))
    elif kind == "Negative":
        ax.add_patch(Arc((0, -0.6), 1.0, 0.7, theta1=20, theta2=160, lw=3, color="#333"))
        ax.plot([-0.6, -0.15], [0.62, 0.52], color="#333", lw=2.5)   # worried brows
        ax.plot([0.6, 0.15], [0.62, 0.52], color="#333", lw=2.5)
    else:
        ax.plot([-0.4, 0.4], [-0.45, -0.45], color="#333", lw=3)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", transparent=True, bbox_inches="tight", pad_inches=0)
    plt.close(fig); buf.seek(0)
    return plt.imread(buf)


FACES = {}
def get_face(kind):
    if kind not in FACES:
        FACES[kind] = face_image(kind)
    return FACES[kind]


def chart_per_source(rows, path):
    rows = sorted(rows, key=lambda r: r["polarity"])
    fig, ax = plt.subplots(figsize=(9, 5.2), dpi=150)
    names = [r["source"] for r in rows]
    pols = [r["polarity"] for r in rows]
    ax.barh(names, pols, color=[COLOR[r["label"]] for r in rows], height=0.6)
    ax.axvline(0, color="#444", lw=1)
    lim = max(0.3, max(abs(p) for p in pols) + 0.15)
    ax.set_xlim(-lim, lim)
    for i, r in enumerate(rows):
        x = r["polarity"] + (0.05 if r["polarity"] >= 0 else -0.05)
        ab = AnnotationBbox(OffsetImage(get_face(r["label"]), zoom=0.22), (x, i),
                            frameon=False, box_alignment=(0.5, 0.5))
        ax.add_artist(ab)
        ax.text(r["polarity"] / 2 if abs(r["polarity"]) > 0.08 else 0.0, i + 0.38,
                f'{r["polarity"]:+.2f}', ha="center", fontsize=8, color="#222")
    ax.set_xlabel("TextBlob polarity  (-1 very negative  to  +1 very positive)")
    ax.set_title("Sentiment by news source - BTC sell-off, Oct 29-30, 2026", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def chart_distribution(rows, path):
    counts = {k: sum(r["label"] == k for r in rows) for k in ("Positive", "Neutral", "Negative")}
    fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
    ax.bar(list(counts), list(counts.values()), color=[COLOR[k] for k in counts], width=0.55)
    top = max(max(counts.values()), 1)
    ax.set_ylim(0, top + 1.6)
    for i, (k, v) in enumerate(counts.items()):
        ax.add_artist(AnnotationBbox(OffsetImage(get_face(k), zoom=0.3), (i, v + 0.7),
                                     frameon=False))
        ax.text(i, v + 0.12, str(v), ha="center", fontweight="bold")
    ax.set_ylabel("Number of sources")
    ax.set_title("Overall sentiment distribution", fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


### Step 5: PDF report builder

In [ ]:
def build_pdf(rows, c1, c2, path):
    ss = getSampleStyleSheet()
    h1 = ParagraphStyle("h1", parent=ss["Title"], fontSize=18, textColor=colors.HexColor("#1a2b4c"))
    h2 = ParagraphStyle("h2", parent=ss["Heading2"], textColor=colors.HexColor("#1a2b4c"), spaceBefore=10)
    body = ParagraphStyle("b", parent=ss["BodyText"], fontSize=10, leading=14)
    small = ParagraphStyle("s", parent=body, fontSize=8.5, leading=11, textColor=colors.HexColor("#444"))
    cell = ParagraphStyle("c", parent=body, fontSize=8, leading=10)

    avg_pol = sum(r["polarity"] for r in rows) / len(rows)
    avg_sub = sum(r["subjectivity"] for r in rows) / len(rows)
    overall = label_for(avg_pol)

    doc = SimpleDocTemplate(path, pagesize=A4, leftMargin=1.6*cm, rightMargin=1.6*cm,
                            topMargin=1.3*cm, bottomMargin=1.3*cm,
                            title="BTC Sell-off Summary and Sentiment Analysis", author="Profit Adda Web")
    s = []
    s += [Paragraph("Bitcoin Sell-off: Summary and Sentiment Analysis", h1),
          Paragraph("Period: Oct 29-30, 2026 &nbsp;|&nbsp; Based on 9 news sources &nbsp;|&nbsp; "
                    "Method: Python + TextBlob", small), Spacer(1, 10)]

    s.append(Paragraph("1. What happened", h2))
    s.append(Paragraph(
        "Bitcoin fell from the 79K-81K range to roughly 76-77K (a five- to six-week low) on "
        "Oct 29-30. The main trigger was a record hack of about 6.3 billion dollars at Binance. "
        "The drop was amplified by about 1.3-1.5 billion dollars of forced liquidations of leveraged "
        "longs, and a hawkish macro backdrop (Fed hold at 3.75-4.00%, high Treasury yields, "
        "ETF outflows) reduced appetite for risk.", body))

    s.append(Paragraph("2. Key causes", h2))
    for t in [
        "<b>Binance hack:</b> about 6.3B dollars stolen; withdrawals paused on some networks; "
        "around 3B frozen; Binance pledged to cover all losses.",
        "<b>Leverage flush:</b> 1.3-1.5B dollars of mostly long positions liquidated in 24 hours.",
        "<b>Macro pressure:</b> Fed held rates in a 10-2 vote, inflation and yields stayed elevated, "
        "and ETF flows were mixed.",
        "<b>Timing debate:</b> some sources say the decline began before the hack news, so the hack "
        "was the trigger but not the only driver.",
    ]:
        s.append(Paragraph("&bull; " + t, body))

    s.append(Paragraph("3. Source-by-source key points", h2))
    data = [["#", "Source", "Key points"]]
    for i, r in enumerate(rows, 1):
        data.append([str(i), Paragraph(r["source"], cell), Paragraph(r["text"], cell)])
    t = Table(data, colWidths=[0.7*cm, 3.2*cm, 13.8*cm], repeatRows=1)
    t.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a2b4c")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTSIZE", (0, 0), (-1, 0), 9),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#bbbbbb")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f3f5f9")]),
    ]))
    s.append(t)
    s.append(Paragraph("Disagreements: hack size estimates ranged from 3.75B to about 6.5B dollars; the "
                       "reported low ranged from about 76.1K to 77.6K depending on the data feed.", small))

    s.append(PageBreak())
    s.append(Paragraph("4. Sentiment analysis (TextBlob)", h2))
    s.append(Paragraph(
        f"Overall average polarity: <b>{avg_pol:+.3f}</b> ({overall}); average subjectivity: "
        f"<b>{avg_sub:.2f}</b>. Labels: polarity above {POS_T} = Positive, below {NEG_T} = Negative, "
        "otherwise Neutral. Face icons: smiling = positive, frowning = negative, flat = neutral.", body))
    s.append(Spacer(1, 6))
    s.append(Image(c1, width=14.5*cm, height=14.5*cm * 5.2 / 9))
    s.append(Spacer(1, 6))
    s.append(Image(c2, width=8*cm, height=8*cm * 4 / 6))

    s.append(Spacer(1, 8))
    tab = [["Source", "Polarity", "Subjectivity", "Label"]]
    for r in rows:
        tab.append([r["source"], f'{r["polarity"]:+.3f}', f'{r["subjectivity"]:.3f}', r["label"]])
    t2 = Table(tab, colWidths=[6*cm, 3*cm, 3.2*cm, 3*cm], rowHeights=0.55*cm)
    style = [("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#1a2b4c")),
             ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
             ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#bbbbbb")),
             ("FONTSIZE", (0, 0), (-1, -1), 8.5), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"), ("ALIGN", (1, 0), (-1, -1), "CENTER")]
    for i, r in enumerate(rows, 1):
        style.append(("TEXTCOLOR", (3, i), (3, i), colors.HexColor(COLOR[r["label"]])))
        style.append(("FONTNAME", (3, i), (3, i), "Helvetica-Bold"))
    t2.setStyle(TableStyle(style))
    s.append(t2)

    s.append(Paragraph("5. Limitations", h2))
    s.append(Paragraph(
        "TextBlob uses a general-purpose word lexicon, not a finance-specific one, so scores are "
        "approximate and often cluster near zero on factual news text. The inputs are condensed "
        "summaries of each source, not full articles. For stronger results, consider a finance-tuned "
        "model such as FinBERT. This report is market context, not investment advice.", body))
    doc.build(s)


### Step 6: run everything

In [ ]:
# Step 6: run the analysis, show results, build the PDF
rows = analyse()

df = pd.DataFrame(rows)[["source", "polarity", "subjectivity", "label"]]
df["emoji"] = df["label"].map(EMOJI)
display(df)

avg = sum(r["polarity"] for r in rows) / len(rows)
print(f"Overall average polarity: {avg:+.3f} -> {EMOJI[label_for(avg)]} {label_for(avg)}")

c1 = os.path.join(OUT_DIR, "sentiment_by_source.png")
c2 = os.path.join(OUT_DIR, "sentiment_distribution.png")
chart_per_source(rows, c1)
chart_distribution(rows, c2)

# show charts inline
from IPython.display import Image as IPImage
display(IPImage(c1)); display(IPImage(c2))

pdf_path = os.path.join(OUT_DIR, "BTC_Selloff_Sentiment_Report.pdf")
build_pdf(rows, c1, c2, pdf_path)
print("PDF saved:", pdf_path)

# download the PDF to your device
from google.colab import files
files.download(pdf_path)
